# 084 — Proyecto: servicio LLM con contratos y evals

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Campos mínimos: `resumen` (string, longitud máxima),
`clausulas_riesgo` (array de objetos {clausula, motivo, severidad enum}),
`confianza` (number 0–1), `requiere_abogado` (boolean); todos `required`.
Política de fallo: 2 reintentos con validación; si falla, respuesta degradada
`{"requiere_abogado": true, "confianza": 0}` sin texto del modelo; todo fallo
queda registrado con la versión de prompt.

**Ejercicio 2.** No desplegar: la F1 de `urgente` cae de 0,88 a 0,71 y es la clase
con mayor costo de error (SLA); la mejora global (+3 netos en 200 casos, McNemar)
no es estadísticamente concluyente. Acciones: ampliar el golden set con ~20 casos
de `urgente` (el actual no la representa) y ajustar v4 específicamente (ejemplos
few-shot de esa clase), luego re-evaluar contra la misma base.

**Ejercicio 3.** El techo práctico es el acuerdo humano-humano (78 %); un juez al
71 % está razonablemente cerca y sirve para *comparar versiones* (señal relativa,
barata, sobre miles de casos). No lo usarías para decisiones absolutas de calidad
("aprobado para producción") ni sobre clases donde su acuerdo sea bajo; y se
re-audita periódicamente porque el juez también cambia.

**Ejercicio 4.** Ejemplo: "la evidencia proviene de un único escenario sintético
con semilla fija; no se midió comportamiento ante entradas adversariales ni carga
concurrente, y ninguna métrica de calidad fue validada por humanos".

In [ ]:
# Ejercicio 1
contrato_salida = {
    "type": "object",
    "properties": {
        "resumen": {"type": "string", "maxLength": 2000},
        "clausulas_riesgo": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "clausula": {"type": "string"},
                    "motivo": {"type": "string"},
                    "severidad": {"type": "string",
                                  "enum": ["alta", "media", "baja"]},
                },
                "required": ["clausula", "motivo", "severidad"],
            },
        },
        "confianza": {"type": "number", "minimum": 0, "maximum": 1},
        "requiere_abogado": {"type": "boolean"},
    },
    "required": ["resumen", "clausulas_riesgo", "confianza", "requiere_abogado"],
}
print(sorted(contrato_salida["required"]))

# Ejercicio 4
result = run_lab("capstone", seed=84)
assert result["kind"] == "capstone"
assert result["evidence"] and result["limitations"]
limitacion_extra = ("evidencia de un solo escenario sintético con semilla fija; "
                    "sin pruebas adversariales ni de carga; sin validación humana")
print(limitacion_extra)
show(result)

## Reflexión

1. ¿Por qué una mejora de accuracy global de 91,5 % a 93,0 % puede ser razón para
   NO desplegar, y qué métrica lo revela?
2. ¿Qué debe registrar cada request para que una regresión detectada el viernes
   sea reproducible el lunes, y cuál de esos campos suele faltar?
3. ¿En qué se parece el contrato `evidence`/`limitations` de los laboratorios de
   este curso al diseño de un servicio LLM honesto?